# Phase 1: Data Understanding

This notebook follows the implementation order in `AGENTS.md` and focuses only on Phase 1:
- transaction data loading
- shapefile loading
- schema and quality checks
- summary report generation


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.src.config import configure_logging, ensure_directories, settings
from backend.src.data_loader import load_all_phase_1_datasets
from backend.src.reporting import (
    profile_geospatial_dataset,
    profile_tabular_dataset,
    write_profiles,
)

configure_logging()
ensure_directories()
PROJECT_ROOT, settings


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'),
 Settings(project_root=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'), backend_root=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/backend'), raw_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data'), interim_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim'), processed_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/processed'), reports_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports'), models_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/models'), logs_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/logs'), backend_logs_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/logs/backend'), frontend_logs_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/logs/frontend'), transaction

In [2]:
datasets = load_all_phase_1_datasets()
datasets.transactions.shape, datasets.property_gdf.shape, datasets.roads_gdf.shape, datasets.facilities_gdf.shape


2026-05-01 18:56:33,541 | INFO | backend.src.data_loader | Loading transaction data from \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\tran_data(AutoRecovered).xlsx
2026-05-01 19:01:27,983 | INFO | backend.src.data_loader | Loading shapefile from \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\ai_usecase_data240326.shp
2026-05-01 19:02:44,631 | INFO | backend.src.data_loader | Loading shapefile from \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\ai_usecase_road240326.shp
2026-05-01 19:02:50,371 | INFO | backend.src.data_loader | Loading shapefile from \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\ai_usecase_facilities240326.shp


((344079, 56), (111497, 29), (10345, 10), (238, 21))

In [3]:
datasets.transactions.head()


,query_year,query_no,book,Deed_No,Deed_Year,Registration_district_code,Registration_district_Name,Registration_RO_code,Registration_Office_Name,Transaction_code,...,Area,Types of area Measurement,Sq ft,Market value per sq ft,setforth_value,market_value,is_bargadar_purchaser,Mouza_Type,Road_Category,Flat_or_Land
0,2023,2001545969,I,5666,2023,23,Paschim Bardhaman,6,A.D.S.R. DURGAPUR,101,...,74.000000,Decimal,32234.400000,0.0,118085056,NaN,NO,RM1,NaN,Land
1,2023,2001650924,I,9047,2023,23,Paschim Bardhaman,6,A.D.S.R. DURGAPUR,201,...,22.960208,Decimal,10001.466792,0.0,0,NaN,NO,DOO,NaN,Land
2,2023,2001655498,I,6186,2023,23,Paschim Bardhaman,6,A.D.S.R. DURGAPUR,101,...,3.992083,Decimal,1738.951546,0.0,0,NaN,NO,DOO,NaN,Land
3,2023,2001499020,I,5853,2023,23,Paschim Bardhaman,6,A.D.S.R. DURGAPUR,101,...,9.629583,Decimal,4194.646525,0.0,0,NaN,NO,DOO,NaN,Land
4,2023,2001552380,I,5656,2023,23,Paschim Bardhaman,6,A.D.S.R. DURGAPUR,101,...,35.731667,Decimal,15564.714032,0.0,0,NaN,NO,DOO,NaN,Land


In [4]:
datasets.property_gdf.head()


,OBJECTID,BLOCK_CODE,BLOCK,PS_CODE,ps_name,Municipali,ADSR_OFFIC,JL_NO,moucode,idn,...,sabek_pt2,CENT_LAT,CENT_LONG,Dist_name,Ward_No,SHAPE_Leng,SHAPE_Area,plot_no,dist_code,geometry
0,1,04,SANKRAIL,09,Sankrail,None,ADSR_RANIHATI,21,021,0504021,...,0,22.557368,88.217131,Howrah,None,0.001499,1.262751e-07,861,05,"POLYGON ((88.21699 22.55754, 88.21711 22.55751..."
1,2,04,SANKRAIL,09,Sankrail,None,ADSR_RANIHATI,21,021,0504021,...,0,22.557429,88.217740,Howrah,None,0.001518,9.285210e-08,863,05,"POLYGON ((88.21755 22.55755, 88.21773 22.55751..."
2,3,04,SANKRAIL,09,Sankrail,None,ADSR_RANIHATI,21,021,0504021,...,0,22.557336,88.214630,Howrah,None,0.001823,2.026836e-07,532,05,"POLYGON ((88.21451 22.55756, 88.21462 22.55754..."
3,4,04,SANKRAIL,09,Sankrail,None,ADSR_RANIHATI,21,021,0504021,...,0,22.557480,88.215423,Howrah,None,0.001016,6.297836e-08,849,05,"POLYGON ((88.21532 22.55761, 88.21542 22.55759..."
4,5,04,SANKRAIL,09,Sankrail,None,ADSR_RANIHATI,21,021,0504021,...,0,22.557529,88.210331,Howrah,None,0.001018,6.541603e-08,452,05,"POLYGON ((88.21032 22.55766, 88.21041 22.55763..."


In [5]:
profiles = [
    profile_tabular_dataset(datasets.transactions, 'transactions'),
    profile_geospatial_dataset(datasets.property_gdf, 'property_layer'),
    profile_geospatial_dataset(datasets.roads_gdf, 'road_layer'),
    profile_geospatial_dataset(datasets.facilities_gdf, 'facilities_layer'),
]

report_json_path, report_md_path = write_profiles(profiles, settings.reports_dir)
report_json_path, report_md_path


2026-05-01 19:03:17,218 | INFO | backend.src.reporting | Saved Phase 1 reports to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\phase_1_data_summary.json and \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\phase_1_data_summary.md


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/phase_1_data_summary.json'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/phase_1_data_summary.md'))

In [6]:
summary = json.loads(Path(report_json_path).read_text())
{
    name: {
        'rows': profile['row_count'],
        'columns': profile['column_count'],
        'duplicates': profile['duplicate_row_count'],
        'candidate_join_keys': profile['candidate_join_keys'][:10],
    }
    for name, profile in summary.items()
}


{'transactions': {'rows': 344079,
  'columns': 56,
  'duplicates': 40653,
  'candidate_join_keys': ['Deed_No',
   'Deed_Year',
   'Registration_district_code',
   'Registration_district_Name',
   'Registration_RO_code',
   'Registration_Office_Name',
   'Transaction_code',
   'Transaction_Name',
   'sl_no_Property',
   'property_district_code']},
 'property_layer': {'rows': 111497,
  'columns': 29,
  'duplicates': 0,
  'candidate_join_keys': ['BLOCK_CODE',
   'PS_CODE',
   'ps_name',
   'moucode',
   'mouza_type',
   'Dist_name',
   'plot_no',
   'dist_code']},
 'road_layer': {'rows': 10345,
  'columns': 10,
  'duplicates': 0,
  'candidate_join_keys': ['R_NAME']},
 'facilities_layer': {'rows': 238,
  'columns': 21,
  'duplicates': 0,
  'candidate_join_keys': ['State_Name', 'District_N', 'Office_Nam']}}

In [7]:
Path(report_md_path).read_text()[:5000]


'# Phase 1 Data Understanding Report\n\nThis report summarizes the transaction and GIS datasets required for Use Case 1.\n\n## transactions\n\n- Type: tabular\n- Rows: 344079\n- Columns: 56\n- Duplicate rows: 40653\n- Candidate join keys: Deed_No, Deed_Year, Registration_district_code, Registration_district_Name, Registration_RO_code, Registration_Office_Name, Transaction_code, Transaction_Name, sl_no_Property, property_district_code, property_district_Name, Property_Office_code, property_Office_Name, ps_code, PS_Name, mouza_code, Mouza_Name, plot_code_type, plot_no, bata_plot_no, premises, Road_code, Road_Name, Zone_no, Special_project_Name, Is_Property_on_Road, Approach_Road_Width, Adjacent_to_Metal_Road, Proposed_Land_use_Code, Proposed_Land_use_Name, Nature_Land_use_Code, Nature_Land_use_Name, Proposed_Apartment_use_Name, Nature_Apartment_use_name, Litigated_Property, Mouza_Type, Road_Category\n\n### Top Missing Columns\n\n- Road_Category: 344079 missing (100.0%)\n- premises: 34102